# Generarea unui Limbaj - Limba Romana

Proiect implementat in SWI-Prolog folosind **DCG (Definite Clause Grammars)**.
DCG este un mecanism Prolog care permite definirea regulilor de productie ale
unei gramatici direct ca clauze, iar Prolog genereaza automat predicatele de
parsare corespunzatoare. Acelasi set de reguli poate fi folosit atat pentru
**verificare** cat si pentru **generare** prin backtracking.

## Descriere

Pornind de la o gramatica formala, sa se genereze expresii corecte gramatical
in limba romana. Gramatica defineste simboluri terminale (cuvinte), simboluri
neterminale (propozitie, sintagma nominala, sintagma verbala etc.) si reguli
de productie care descriu cum se construiesc structurile sintactice.

## Solutie

Acordul gramatical (gen, numar, persoana) este propagat prin atributele
regulilor DCG — subiectul transmite numarul si persoana catre predicat,
iar articolul transmite genul si numarul catre substantiv si adjectiv.

## Structura Proiectului

- **gramatica.pl** - lexiconul (terminale) si regulile DCG (neterminale); propaga atributele de acord
- **verificare.pl** - predicatul `propozitie/1` care verifica daca o lista de cuvinte e valida
- **generare.pl** - predicatele `genereaza/1`, `genereaza_random/1`, `afiseaza_n/1` pentru generarea propozitiilor
- **teste.pl** - teste manuale organizate pe categorii: verificare, acord gramatical, pronume, complemente, adverbe, edge-case

## Gramatica

Regulile de productie principale:
```
propozitie   --> sintagma_nom(Numar, Persoana), sintagma_verb(Numar, Persoana)
sintagma_nom --> articol, substantiv
sintagma_nom --> articol, substantiv, adjectiv
sintagma_nom --> pronume
sintagma_verb --> verb
sintagma_verb --> verb, adverb
sintagma_verb --> verb, sintagma_nom
sintagma_verb --> verb, sintagma_nom, adverb
sintagma_verb --> verb, complement
sintagma_verb --> verb, sintagma_nom, complement
complement   --> prepozitie, sintagma_nom
```

Atributele `Numar` si `Persoana` propaga acordul subiect-predicat.

## Incarcare fisiere

Incarcam cele patru module principale ale proiectului:
- `gramatica.pl` — regulile DCG si lexiconul
- `verificare.pl` — predicatul de verificare
- `generare.pl` — predicatele de generare
- `teste.pl` — testele manuale organizate pe categorii

In [1]:
:- ensure_loaded('../cod/gramatica.pl').
:- ensure_loaded('../cod/verificare.pl').
:- ensure_loaded('../cod/generare.pl').
:- ensure_loaded('../cod/teste.pl').

## Verificare propozitii

Predicatul `propozitie/1` verifica daca o lista de cuvinte respecta gramatica.
Intern apeleaza regula DCG `propozitie//0` cu lista vida ca diferenta de liste.

In [2]:
% Propozitie valida simpla: articol + substantiv + verb la persoana 3 singular
propozitie([un, baiat, citeste]).

true

Acordul subiect-verb functioneaza si pentru pronume personale.
Pronumele `eu` impune persoana 1 singular, deci verbul trebuie sa fie `citesc`.

In [3]:
% Acord subiect-verb: persoana 1 singular
propozitie([eu, citesc]).

true

Acordul articol-substantiv este verificat strict.
Articolul `o` este feminin, deci nu poate insoti substantivul masculin `baiat`.

In [4]:
% Acord gresit: articol feminin + substantiv masculin -> false
propozitie([o, baiat, citeste]).

false

## Generare propozitii

Predicatul `genereaza/1` produce propozitii valide prin backtracking pe regulile DCG.
Prolog exploreaza toate combinatiile posibile de terminale care satisfac regulile,
returnand cate o propozitie la fiecare pas.

In [5]:
% Prima propozitie generata prin backtracking
genereaza(P).

P = [un,baiat,citeste]

Predicatul `afiseaza_n/1` genereaza N propozitii alese aleator din toate
propozitiile posibile si le afiseaza formatat, cu punct la sfarsit.

In [6]:
% 5 propozitii aleatorii
afiseaza_n(5).

un baiat citeste un baiat mare.
un baiat scrie o floare.
un baiat citeste niste copii buni.
un baiat scrie un prieten destept.
un baiat citeste niste carti mari.

true

## Teste manuale (teste.pl)

Predicatul `ruleaza_teste/0` ruleaza toate testele organizate pe categorii:
verificare de baza, acord gramatical, pronume personale, complemente, adverbe
si edge-case. Fiecare test afiseaza `PASS` sau `FAIL`.

In [7]:
% Ruleaza toate testele din teste.pl
ruleaza_teste.

=== Teste Verificare ===
  PASS: un baiat citeste
  PASS: o fata scrie
  PASS: un copil merge la o scoala
  PASS: o fata frumoasa citeste o carte
  PASS (corect invalid): un fata citeste
  PASS (corect invalid): o baiat scrie
  PASS (corect invalid): citeste un baiat
  PASS (corect invalid): 

=== Teste Acord Gramatical ===
  PASS: un om merge
  PASS: o masa merge
  PASS: niste baieti citesc
  PASS: niste fete scriu
  PASS (corect invalid): un casa merge
  PASS (corect invalid): o copil citeste
  PASS: un baiat frumos merge
  PASS: o fata frumoasa merge
  PASS: niste baieti frumosi merg
  PASS (corect invalid): un baiat frumoasa merge
  PASS (corect invalid): o fata frumos scrie
  PASS: un baiat citeste
  PASS: niste baieti citesc
  PASS (corect invalid): un baiat citesc
  PASS (corect invalid): niste fete scrie

=== Teste Pronume Personale ===
  PASS: eu citesc
  PASS: eu merg la o scoala
  PASS: eu vad un prieten
  PASS: tu citesti
  PASS: tu mergi acasa
  PASS: tu scrii o carte
  PA

true

## Teste automate (plunit)

Testele sunt organizate in doua categorii:
- **Verificare** — propozitii valide care trebuie sa returneze `true` si propozitii
  invalide care trebuie sa returneze `false`
- **Generare** — verifica ca predicatele de generare produc rezultate non-vide
  si respecta limitele impuse

In [8]:
:- begin_tests(proiect).

:- ensure_loaded('../cod/gramatica.pl').
:- ensure_loaded('../cod/verificare.pl').
:- ensure_loaded('../cod/generare.pl').
:- ensure_loaded('../cod/teste.pl').

test(propozitie_valida_simpla) :-
    propozitie([un, baiat, citeste]).

test(propozitie_valida_cu_complement) :-
    propozitie([un, copil, merge, la, o, scoala]).

test(propozitie_valida_cu_adjectiv) :-
    propozitie([o, fata, frumoasa, citeste, o, carte]).

test(propozitie_invalida_ordine, [fail]) :-
    propozitie([citeste, un, baiat]).

test(propozitie_invalida_goala, [fail]) :-
    propozitie([]).

test(propozitie_invalida_acord, [fail]) :-
    propozitie([un, fata, citeste]).

test(genereaza_o_propozitie) :-
    genereaza(P),
    assertion(P \= []).

test(genereaza_maxim_5) :-
    genereaza_max(5, Lista),
    length(Lista, L),
    assertion(L =< 5).

test(genereaza_random) :-
    genereaza_random(P),
    assertion(P \= []).

test(genereaza_n) :-
    genereaza_n(3, Propozitii),
    length(Propozitii, L),
    assertion(L =< 3).

:- end_tests(proiect).

:- run_tests.

% Defined test unit proiect

% Start unit: proiect
% [1/10] proiect:propozitie_valida_simpla ..
% [2/10] proiect:propoziti..alida_cu_complement ..
% [3/10] proiect:propozitie_valida_cu_adjectiv ..
% [4/10] proiect:propozitie_invalida_ordine ......... passed (0.000 sec)
% [5/10] proiect:propozitie_invalida_goala .......... passed (0.000 sec)
% [6/10] proiect:propozitie_invalida_acord .......... passed (0.000 sec)
% [7/10] proiect:genereaza_o_propozitie ..
% [8/10] proiect:genereaza_maxim_5 .................. passed (0.000 sec)
% [9/10] proiect:genereaza_random ................... passed (0.000 sec)
% [10/10] proiect:genereaza_n ....................... passed (0.000 sec)
% End unit proiect: passed (0.000 sec CPU)
% All 10 tests passed in 0.019 seconds (0.000 cpu)

true